# 🎵 Description Model — Music Recommender

Recommends songs based on a **free-text natural language description** of what the user wants to listen to.

---

## How It Works

1. Songs are represented as two parallel embedding spaces:
   - **Embedding 1** — Built from `"{genre} song by {artist}. Lyrics: {preview}"` — anchors on explicit content
   - **Embedding 2** — Built from the pre-generated `full_description` column — captures broader mood
2. At query time, the user's input is encoded and compared against both spaces via cosine similarity
3. Three retrieval strategies are available: lyrics-only, description-only, or combined
4. A Gradio chatbot provides the user interface

---

## Dataset

- **Required:** `60k-songs-with-description` from Kaggle
- File: `/kaggle/input/60k-songs-with-description/songs_with_descriptions_V3.csv`

## 0. Install Dependencies

In [ ]:
!pip install -q langdetect gradio sentence-transformers transformers scikit-learn

## 1. Setup — Paths and Imports

In [ ]:
import sys
import os

# Add the project root to the Python path so `src` imports work
PROJECT_ROOT = "/kaggle/working/description_model"
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
from pathlib import Path

# Kaggle paths
DATA_CSV  = "/kaggle/input/60k-songs-with-description/songs_with_descriptions_V3.csv"
MODEL_DIR = Path("/kaggle/working/models")
MODEL_DIR.mkdir(exist_ok=True)

print("Python path set. Ready to import src modules.")

## 2. Load and Preprocess Data

In [ ]:
from src.preprocessing import preprocess_dataframe

raw_df = pd.read_csv(DATA_CSV)
print(f"Loaded {len(raw_df):,} songs. Columns: {list(raw_df.columns)}")

df = preprocess_dataframe(raw_df)
print(f"Preprocessing complete. New columns: description_1, description_2")
df[['song', 'artist', 'translated_genres', 'description_1']].head(3)

## 3. Generate Embeddings

> ⚠️ This step takes ~15–30 minutes on 60k songs. Run once and save — the cells below reload from disk automatically.

In [ ]:
from src.training import load_model, build_and_save_embeddings, load_embeddings

EMBEDDINGS_PATH_1 = MODEL_DIR / "song_embeddings_1.npy"
EMBEDDINGS_PATH_2 = MODEL_DIR / "song_embeddings_2.npy"

embed_model = load_model()  # all-MiniLM-L6-v2

if EMBEDDINGS_PATH_1.exists() and EMBEDDINGS_PATH_2.exists():
    print("Found existing embeddings — loading from disk.")
    embeddings_1, embeddings_2 = load_embeddings(MODEL_DIR)
else:
    print("No cached embeddings found — generating now (this takes a while)...")
    embeddings_1, embeddings_2 = build_and_save_embeddings(df, embed_model, MODEL_DIR)

print(f"embeddings_1 shape: {embeddings_1.shape}")
print(f"embeddings_2 shape: {embeddings_2.shape}")

## 4. Build Genre Vocabulary

In [ ]:
from src.utils import build_genre_list

genre_list = build_genre_list(df)
print(f"Unique genres: {len(genre_list)}")
print(genre_list[:20])

## 5. Quick Recommendation Tests

In [ ]:
from src.inference import (
    recommend_by_lyrics_description,
    recommend_by_full_description,
    recommend_by_combined,
)
from src.utils import add_youtube_links

QUERY = "a grunge song about pain and toxic love"
print(f"Query: {QUERY}\n")

print("--- Method 1: Lyrics Embedding ---")
result_1 = recommend_by_lyrics_description(df, embeddings_1, embed_model, QUERY, genre_list)
print(add_youtube_links(result_1))

print("\n--- Method 2: Description Matching ---")
result_2 = recommend_by_full_description(df, embeddings_2, embed_model, QUERY, genre_list)
print(add_youtube_links(result_2))

print("\n--- Method 3: Combined (Recommended) ---")
result_3 = recommend_by_combined(df, embeddings_1, embeddings_2, embed_model, QUERY, genre_list)
print(add_youtube_links(result_3))

## 6. Launch the Gradio Chatbot

In [ ]:
from src.inference.app import build_app

app = build_app(df, embeddings_1, embeddings_2, embed_model, genre_list)
app.launch(share=True)  # share=True gives a public ngrok URL on Kaggle